In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import os

BASE_DIR = "/content/drive/MyDrive/EMBED"

CSV_PATH = os.path.join(BASE_DIR, "embed_future_risk_dataset_512.csv")

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print("Unique patients:", df["empi_anon"].nunique())

df.head()

Dataset shape: (1038, 13)
Unique patients: 40


,empi_anon,acc_anon,study_date_anon,ViewPosition,ImageLateralityFinal,processed_image_path,future_risk_label,days_to_cancer,risk_1yr,risk_2yr,risk_3yr,risk_4yr,risk_5yr
0,57769289,3512912135438605,2016-06-15 00:00:00,CC,R,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1
1,57769289,3504436559522696,2017-01-21 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1
2,57769289,3504436559522696,2017-01-21 00:00:00,CC,L,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1
3,57769289,3512912135438605,2016-06-15 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1
4,57769289,6528636244388176,2016-07-20 00:00:00,CC,L,/content/drive/MyDrive/EMBED/processed_images_...,1,412.0,0,1,1,1,1


In [4]:
risk_cols = ["risk_1yr", "risk_2yr", "risk_3yr", "risk_4yr", "risk_5yr"]

patient_df = (
    df.groupby("empi_anon")[risk_cols + ["future_risk_label"]]
    .max()
    .reset_index()
)

print("Patient-level future label count:")
print(patient_df["future_risk_label"].value_counts())

print("\nPatient-level year-wise counts:")
print(patient_df[risk_cols].sum())

Patient-level future label count:
future_risk_label
0    20
1    20
Name: count, dtype: int64

Patient-level year-wise counts:
risk_1yr     3
risk_2yr     9
risk_3yr    14
risk_4yr    16
risk_5yr    18
dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

patients = patient_df["empi_anon"]
labels = patient_df["future_risk_label"]

train_patients, temp_patients = train_test_split(
    patients,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

temp_df = patient_df[patient_df["empi_anon"].isin(temp_patients)]

val_patients, test_patients = train_test_split(
    temp_df["empi_anon"],
    test_size=0.50,
    random_state=42,
    stratify=temp_df["future_risk_label"]
)

print("Train patients:", len(train_patients))
print("Val patients:", len(val_patients))
print("Test patients:", len(test_patients))

Train patients: 28
Val patients: 6
Test patients: 6


In [6]:
train_df = df[df["empi_anon"].isin(train_patients)].copy()
val_df = df[df["empi_anon"].isin(val_patients)].copy()
test_df = df[df["empi_anon"].isin(test_patients)].copy()

print("Train images:", train_df.shape)
print("Val images:", val_df.shape)
print("Test images:", test_df.shape)

print("\nTrain patients:", train_df["empi_anon"].nunique())
print("Val patients:", val_df["empi_anon"].nunique())
print("Test patients:", test_df["empi_anon"].nunique())

Train images: (697, 13)
Val images: (168, 13)
Test images: (173, 13)

Train patients: 28
Val patients: 6
Test patients: 6


In [7]:
SPLIT_DIR = os.path.join(BASE_DIR, "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

train_df.to_csv(os.path.join(SPLIT_DIR, "train_512.csv"), index=False)
val_df.to_csv(os.path.join(SPLIT_DIR, "val_512.csv"), index=False)
test_df.to_csv(os.path.join(SPLIT_DIR, "test_512.csv"), index=False)

print("Saved train, val, and test CSV files.")
print(SPLIT_DIR)

Saved train, val, and test CSV files.
/content/drive/MyDrive/EMBED/splits


In [8]:
!pip install -q torch torchvision opencv-python

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [10]:
class EMBEDFutureRiskDataset(Dataset):
    def __init__(self, dataframe, risk_cols, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.risk_cols = risk_cols
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = row["processed_image_path"]
        image = Image.open(image_path).convert("L")

        labels = row[self.risk_cols].values.astype("float32")
        labels = torch.tensor(labels, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, labels

In [14]:
train_dataset = EMBEDFutureRiskDataset(train_df, risk_cols, train_transform)
val_dataset = EMBEDFutureRiskDataset(val_df, risk_cols, val_test_transform)
test_dataset = EMBEDFutureRiskDataset(test_df, risk_cols, val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2
)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:")
print(labels)

Image batch shape: torch.Size([8, 3, 512, 512])
Label batch shape: torch.Size([8, 5])
Labels:
tensor([[0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])
